In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
import numpy as np

NASA_PATH = "drive/MyDrive/datasets/nasa_power_daily_raw.csv"
MET_PATH = "drive/MyDrive/datasets/met_office_london_temp_daily_2016_2022.csv"

print("NASA exists:", NASA_PATH)
print("MET exists:", MET_PATH)



NASA exists: drive/MyDrive/datasets/nasa_power_daily_raw.csv
MET exists: drive/MyDrive/datasets/met_office_london_temp_daily_2016_2022.csv


In [8]:
nasa = pd.read_csv(NASA_PATH)
met = pd.read_csv(MET_PATH)

# parse dates
nasa["date"] = pd.to_datetime(nasa["date"])
met["date"] = pd.to_datetime(met["date"])

# keep London only
nasa_london = nasa[nasa["city"] == "London"].copy()

print("NASA London rows:", len(nasa_london))
print("Met rows:", len(met))
print("Met stations:", met["station"].nunique(), sorted(met["station"].unique()))

nasa_london.head()


NASA London rows: 2557
Met rows: 7649
Met stations: 3 ['Heathrow', 'Kew Gardens', 'Northolt']


,city,latitude,longitude,date,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M
0,London,51.5074,-0.1278,2016-01-01,5.17,7.09,2.47,5.18,94.66,3.94
1,London,51.5074,-0.1278,2016-01-02,8.58,9.52,6.72,3.50,95.32,5.20
2,London,51.5074,-0.1278,2016-01-03,6.55,8.29,4.25,11.59,96.01,4.50
3,London,51.5074,-0.1278,2016-01-04,7.05,8.49,5.85,6.88,95.86,3.68
4,London,51.5074,-0.1278,2016-01-05,7.32,9.28,5.91,3.12,96.07,2.04


In [9]:
# Keep only what we need
nasa_london = nasa_london[["date", "T2M_MAX", "T2M_MIN"]].copy()
met = met[["station", "date", "tmax_c", "tmin_c"]].copy()

# Rename to common names
nasa_london.rename(columns={"T2M_MAX": "tmax_nasa", "T2M_MIN": "tmin_nasa"}, inplace=True)
met.rename(columns={"tmax_c": "tmax_met", "tmin_c": "tmin_met"}, inplace=True)

nasa_london.head(), met.head()


(        date  tmax_nasa  tmin_nasa
 0 2016-01-01       7.09       2.47
 1 2016-01-02       9.52       6.72
 2 2016-01-03       8.29       4.25
 3 2016-01-04       8.49       5.85
 4 2016-01-05       9.28       5.91,
     station       date  tmax_met  tmin_met
 0  Heathrow 2016-01-01       8.6       1.3
 1  Heathrow 2016-01-02      11.3       7.8
 2  Heathrow 2016-01-03      10.1       4.9
 3  Heathrow 2016-01-04      10.9       5.7
 4  Heathrow 2016-01-05      10.1       5.1)

In [10]:
joined = met.merge(nasa_london, on="date", how="inner")

print("Joined rows:", len(joined))
print("Stations:", joined["station"].nunique(), sorted(joined["station"].unique()))
print("Date range:", joined["date"].min().date(), "to", joined["date"].max().date())

joined.head()


Joined rows: 7649
Stations: 3 ['Heathrow', 'Kew Gardens', 'Northolt']
Date range: 2016-01-01 to 2022-12-31


,station,date,tmax_met,tmin_met,tmax_nasa,tmin_nasa
0,Heathrow,2016-01-01,8.6,1.3,7.09,2.47
1,Heathrow,2016-01-02,11.3,7.8,9.52,6.72
2,Heathrow,2016-01-03,10.1,4.9,8.29,4.25
3,Heathrow,2016-01-04,10.9,5.7,8.49,5.85
4,Heathrow,2016-01-05,10.1,5.1,9.28,5.91


In [11]:
def mean_bias(pred, obs):
    # pred - obs
    return float(np.mean(pred - obs))

def rmse(pred, obs):
    return float(np.sqrt(np.mean((pred - obs) ** 2)))

def corr(pred, obs):
    return float(np.corrcoef(pred, obs)[0, 1])


In [12]:
rows = []

for station, g in joined.groupby("station"):
    g = g.dropna()

    rows.append({
        "scope": station,
        "n_days": len(g),
        "bias_tmax": mean_bias(g["tmax_nasa"], g["tmax_met"]),
        "rmse_tmax": rmse(g["tmax_nasa"], g["tmax_met"]),
        "corr_tmax": corr(g["tmax_nasa"], g["tmax_met"]),
        "bias_tmin": mean_bias(g["tmin_nasa"], g["tmin_met"]),
        "rmse_tmin": rmse(g["tmin_nasa"], g["tmin_met"]),
        "corr_tmin": corr(g["tmin_nasa"], g["tmin_met"]),
    })

# overall (all stations pooled)
g = joined.dropna()
rows.append({
    "scope": "overall_pooled",
    "n_days": len(g),
    "bias_tmax": mean_bias(g["tmax_nasa"], g["tmax_met"]),
    "rmse_tmax": rmse(g["tmax_nasa"], g["tmax_met"]),
    "corr_tmax": corr(g["tmax_nasa"], g["tmax_met"]),
    "bias_tmin": mean_bias(g["tmin_nasa"], g["tmin_met"]),
    "rmse_tmin": rmse(g["tmin_nasa"], g["tmin_met"]),
    "corr_tmin": corr(g["tmin_nasa"], g["tmin_met"]),
})

metrics = pd.DataFrame(rows).sort_values("scope")
metrics


,scope,n_days,bias_tmax,rmse_tmax,corr_tmax,bias_tmin,rmse_tmin,corr_tmin
0,Heathrow,2557,-1.386371,1.851309,0.983669,-1.470282,2.008105,0.965937
1,Kew Gardens,2533,-1.288974,1.823913,0.980914,-0.587177,1.842228,0.946372
2,Northolt,2557,-1.101545,1.669000,0.982236,-0.675835,1.842046,0.950619
3,overall_pooled,7647,-1.258869,1.783085,0.982083,-0.912115,1.899248,0.951500


In [13]:
joined["tmax_error"] = joined["tmax_nasa"] - joined["tmax_met"]
joined["tmin_error"] = joined["tmin_nasa"] - joined["tmin_met"]

summary = joined.groupby("station")[["tmax_error", "tmin_error"]].agg(["mean", "std", "median"])
summary


tmax_error                  tmin_error                 
                  mean       std median       mean       std median
station                                                            
Heathrow     -1.386371  1.227155  -1.41  -1.470282  1.368024  -1.47
Kew Gardens  -1.289767  1.291047  -1.33  -0.587177  1.746491  -0.69
Northolt     -1.101545  1.254103  -1.14  -0.675835  1.713922  -0.89

In [14]:
out_metrics = "drive/MyDrive/uk_climate_anomaly_outputs/nasa_metoffice_validation_metrics.csv"
metrics.to_csv(out_metrics, index=False)
print("Saved:", out_metrics)


Saved: drive/MyDrive/uk_climate_anomaly_outputs/nasa_metoffice_validation_metrics.csv
